In [2]:
import pandas as pd
import numpy as np
import json

In [9]:
# Load your datasets
trains_df = pd.read_csv('trains.csv')
schedules_df = pd.read_csv('schedules.csv') 
stations_df = pd.read_csv('stations.csv')

In [10]:
# Check what's actually missing
print("Missing values per column:")
print("trains dataset:")
print(trains_df.isnull().sum())
print("schedules dataset:")
print(schedules_df.isnull().sum())
print("stations dataset:")
print(stations_df.isnull().sum())

Missing values per column:
trains dataset:
train_number            0
train_name              1
train_type             15
from_station_code       0
from_station_name       0
to_station_code         0
to_station_name         0
departure_time         15
arrival_time           15
duration_hours         15
duration_minutes       15
distance_km            15
zone                   15
first_class             0
first_ac                0
second_ac               0
third_ac                0
chair_car               0
sleeper                 0
classes              5208
return_train          599
route_coordinates       0
dtype: int64
schedules dataset:
arrival_time      27835
day               22561
train_name            8
station_name          2
station_code          0
record_id             0
train_number          0
departure_time    27827
dtype: int64
stations dataset:
station_code       0
station_name       1
state           4593
zone            4532
address         4593
coordinates        0
dtyp

In [11]:
def create_final_scenario():
    """Final working solution using your real data"""
    trains = pd.read_csv('trains.csv')
    schedules = pd.read_csv('schedules.csv')
    
    # Get real trains from NDLS
    ndls_trains = schedules[schedules['station_code'] == 'NDLS'].head(8)
    
    print("🎯 USING REAL TRAINS FROM YOUR DATASET:")
    real_trains = []
    base_times = ['15:30', '15:45', '16:00', '16:15', '16:15', '16:30', '17:00', '17:30']
    
    for i, (idx, train) in enumerate(ndls_trains.iterrows()):
        # Get capacity data from trains.csv
        train_capacity = trains[trains['train_number'] == train['train_number']]
        
        if not train_capacity.empty:
            cap_data = train_capacity.iloc[0]
            # Calculate passenger load from REAL data
            passenger_load = (
                (cap_data.get('first_ac', 0) or 0) * 18 +
                (cap_data.get('second_ac', 0) or 0) * 48 + 
                (cap_data.get('third_ac', 0) or 0) * 72 +
                (cap_data.get('sleeper', 0) or 0) * 72
            ) * 0.7  # 70% occupancy
            
            # Ensure reasonable values
            if passenger_load < 100:
                passenger_load = 1200 if i in [3, 4, 6] else 800
        else:
            passenger_load = 1200 if i in [3, 4, 6] else 800
        
        train_data = {
            'train_id': i + 1,
            'train_number': str(train['train_number']),  # Convert to string for JSON
            'train_name': str(train.get('train_name', f"Train {train['train_number']}")),
            'train_type': 'Special' if i in [3, 4, 6] else 'Express',
            'scheduled_arrival': base_times[i],
            'scheduled_departure': f"{int(base_times[i][:2])+1}:{base_times[i][3:]}",
            'passenger_load': int(passenger_load),  # Convert to Python int
            'priority': 'High' if i in [3, 4, 6] else 'Normal',
            'original_platform': [1, 4, 4, 4, 6, 2, 3, 5][i],
            'current_eta': ['15:30', '15:45', '16:40', '16:05', '15:55', '16:30', '17:00', '17:30'][i],
            'status': ['OnTime', 'OnTime', 'Delayed', 'Early', 'Early', 'OnTime', 'OnTime', 'OnTime'][i]
        }
        
        real_trains.append(train_data)
        print(f"  🚂 {train_data['train_name']} ({train_data['train_number']}): {train_data['passenger_load']} passengers")
    
    return real_trains

def main():
    print("🚆 FINAL DIWALI RUSH SCENARIO")
    print("=" * 60)
    
    # Create scenario with REAL data
    trains_data = create_final_scenario()
    
    # Create platforms data
    platforms_data = []
    platform_info = [
        {'id': 1, 'capacity': 1500, 'occupancy': 300},
        {'id': 2, 'capacity': 1200, 'occupancy': 200},
        {'id': 3, 'capacity': 1800, 'occupancy': 800},
        {'id': 4, 'capacity': 1600, 'occupancy': 1100},
        {'id': 5, 'capacity': 1400, 'occupancy': 600},
        {'id': 6, 'capacity': 1300, 'occupancy': 400},
        {'id': 7, 'capacity': 1100, 'occupancy': 100},
        {'id': 8, 'capacity': 1000, 'occupancy': 200}
    ]
    
    for platform in platform_info:
        platforms_data.append({
            'platform_id': platform['id'],
            'capacity': platform['capacity'],
            'is_operational': True,
            'proximity_to_concourse': 'A' if platform['id'] <= 2 else 'B' if platform['id'] <= 5 else 'C',
            'has_handicap_access': platform['id'] % 2 == 1,  # Odd platforms have access
            'current_occupancy': platform['occupancy']
        })
    
    # Prepare output with Python native types
    output = {
        'trains': trains_data,
        'platforms': platforms_data,
        'metadata': {
            'scenario': 'Diwali Rush at New Delhi Station',
            'total_passengers': sum(t['passenger_load'] for t in trains_data),
            'conflict_platforms': [4],  # Platform 4 has multiple trains
            'crowded_platforms': [3, 4, 5]  # Platforms 3-5 are crowded
        }
    }
    
    # Save to JSON
    with open('live_scenario.json', 'w') as f:
        json.dump(output, f, indent=2)
    
    print("\n" + "=" * 60)
    print("✅ SCENARIO CREATED SUCCESSFULLY!")
    print("=" * 60)
    print(f"📊 {len(trains_data)} REAL trains from your dataset")
    print(f"👥 Total passengers: {output['metadata']['total_passengers']}")
    print(f"🚨 Platform 4 has {len([t for t in trains_data if t['original_platform'] == 4])} trains assigned!")
    print(f"🔥 Crowded platforms: 3, 4, 5")
    print(f"💾 Saved to: live_scenario.json")
    
    # Show the conflict scenario
    print("\n🎯 CONFLICT SCENARIO CREATED:")
    platform_assignments = {}
    for train in trains_data:
        platform = train['original_platform']
        if platform not in platform_assignments:
            platform_assignments[platform] = []
        platform_assignments[platform].append(train['train_name'])
    
    for platform, trains in platform_assignments.items():
        if len(trains) > 1:
            print(f"  🚨 Platform {platform}: {', '.join(trains)}")
        else:
            print(f"  ✅ Platform {platform}: {trains[0]}")

if __name__ == "__main__":
    main()

🚆 FINAL DIWALI RUSH SCENARIO
🎯 USING REAL TRAINS FROM YOUR DATASET:
  🚂 New Delhi-Sealdah AC Duronto Express (12260): 800 passengers
  🚂 Amritsar Shatabdi Express (12031): 800 passengers
  🚂 Dehradun Jan Shatabdi Express (12055): 800 passengers
  🚂 NewDelhi-Agra InterCity Express (14212): 1200 passengers
  🚂 Magadh Express (12402): 147 passengers
  🚂 Vaishali SF Express (12554): 147 passengers
  🚂 New Delhi-Rohtak Express (14323): 1200 passengers
  🚂 New Delhi-Ranchi Rajdhani Express (12454): 800 passengers

✅ SCENARIO CREATED SUCCESSFULLY!
📊 8 REAL trains from your dataset
👥 Total passengers: 5894
🚨 Platform 4 has 3 trains assigned!
🔥 Crowded platforms: 3, 4, 5
💾 Saved to: live_scenario.json

🎯 CONFLICT SCENARIO CREATED:
  ✅ Platform 1: New Delhi-Sealdah AC Duronto Express
  🚨 Platform 4: Amritsar Shatabdi Express, Dehradun Jan Shatabdi Express, NewDelhi-Agra InterCity Express
  ✅ Platform 6: Magadh Express
  ✅ Platform 2: Vaishali SF Express
  ✅ Platform 3: New Delhi-Rohtak Express
 